In [1]:
import pennylane as qml
import numpy as np

In [2]:
n_data = 9
n_anc = 8
n_qubits = n_data + n_anc

data_qubits = list(range(n_data))
anc_z = list(range(n_data, n_data + 4))
anc_x = list(range(n_data + 4, n_qubits))

DATA_WIRES = data_qubits
ANC_WIRES = anc_z + anc_x

# 각 stabilizer가 어떤 data qubit에 연결되는지
X_stabilizers = [
    [0,1],
    [1,2,4,5],
    [3,4,6,7],
    [7,8]
]

Z_stabilizers = [
    [0,1,3,4],
    [2,5],
    [3,6],
    [4,5,7,8],
]

In [3]:
# --------------------------------------------------
# Pauli utilities
# --------------------------------------------------

def apply_pauli(pauli_type, wires):
    if pauli_type == "X":
        qml.PauliX(wires=wires)
    elif pauli_type == "Y":
        qml.PauliY(wires=wires)
    elif pauli_type == "Z":
        qml.PauliZ(wires=wires)
    else:
        raise ValueError(f"Unknown Pauli type: {pauli_type}")


def apply_pauli_error(error_list, error_wires):
    for p, w in zip(error_list, error_wires):
        apply_pauli(p, w)


In [4]:
def get_faults_for_round(fault_schedule, current_round):
    """
    fault_schedule 예시:

    [
        {"round": 1, "wires": [0, 9], "error": ["X", "Z"]},
        {"round": 3, "wires": [4, 12], "error": ["X", "X"]},
    ]
    """
    return [
        fault for fault in fault_schedule
        if fault["round"] == current_round
    ]


def inject_fault_if_matches_gate(gate_wires, current_faults):
    """
    현재 CNOT gate의 두 wire가 fault["wires"]와 같으면
    해당 correlated Pauli error를 삽입.
    """

    for fault in current_faults:
        if sorted(gate_wires) == sorted(fault["wires"]):
            apply_pauli_error(
                error_list=fault["error"],
                error_wires=fault["wires"],
            )

In [5]:
# --------------------------------------------------
# Z stabilizer measurement
# --------------------------------------------------

def measure_Z_stabilizer_with_fault_schedule(
    anc,
    qubits,
    current_faults,
):
    """
    Z stabilizer 측정.

    기존 코드와 동일:
        data q -> ancilla CNOT
    """

    for q in qubits:
        qml.CNOT(wires=[q, anc])

        inject_fault_if_matches_gate(
            gate_wires=[q, anc],
            current_faults=current_faults,
        )

    m = qml.measure(anc, reset=True)
    return m


# --------------------------------------------------
# X stabilizer measurement
# --------------------------------------------------

def measure_X_stabilizer_with_fault_schedule(
    anc,
    qubits,
    current_faults,
):
    """
    X stabilizer 측정.

    기존 코드와 동일:
        H on anc
        ancilla -> data q CNOT
        H on anc
    """

    qml.Hadamard(wires=anc)

    for q in qubits:
        qml.CNOT(wires=[anc, q])

        inject_fault_if_matches_gate(
            gate_wires=[anc, q],
            current_faults=current_faults,
        )

    qml.Hadamard(wires=anc)

    m = qml.measure(anc, reset=True)
    return m

In [7]:
def prepare_logical_zero():
    """
    Prepare |0_L> of the d=3 rotated surface code
    with logical Z_L = Z0 Z1 Z2.
    """

    # Independent variables:
    # a -> qubit 0
    # b -> qubit 2
    # c -> qubit 3
    # d -> qubit 8
    qml.Hadamard(wires=0)
    qml.Hadamard(wires=2)
    qml.Hadamard(wires=3)
    qml.Hadamard(wires=8)

    # q1 = a ⊕ b
    qml.CNOT(wires=[0, 1])
    qml.CNOT(wires=[2, 1])

    # q4 = b ⊕ c
    qml.CNOT(wires=[2, 4])
    qml.CNOT(wires=[3, 4])

    # q5 = b
    qml.CNOT(wires=[2, 5])

    # q6 = c
    qml.CNOT(wires=[3, 6])

    # q7 = c ⊕ d
    qml.CNOT(wires=[3, 7])
    qml.CNOT(wires=[8, 7])

In [8]:
# --------------------------------------------------
# One syndrome extraction round
# --------------------------------------------------

def syndrome_round_with_fault_schedule(
    current_round,
    fault_schedule,
):
    """
    한 round에서 8개 syndrome 측정.

    반환 순서:
        [Z0, Z1, Z2, Z3, X0, X1, X2, X3]
    """

    current_faults = get_faults_for_round(
        fault_schedule=fault_schedule,
        current_round=current_round,
    )

    round_record = []

    for i, stab in enumerate(Z_stabilizers):
        m = measure_Z_stabilizer_with_fault_schedule(
            anc=anc_z[i],
            qubits=stab,
            current_faults=current_faults,
        )
        round_record.append(m)

    for i, stab in enumerate(X_stabilizers):
        m = measure_X_stabilizer_with_fault_schedule(
            anc=anc_x[i],
            qubits=stab,
            current_faults=current_faults,
        )
        round_record.append(m)

    return round_record


In [20]:
n_rounds = 4
shots = 1

In [21]:
# --------------------------------------------------
# Repeated syndrome circuit
# --------------------------------------------------

@qml.qnode(qml.device("default.qubit", wires=n_qubits, shots=shots))
def repeated_syndrome_with_fault_schedule(
    fault_schedule=[],
    initial_error_list=[],
    initial_error_wires=[],
):
    measurement_record = []

    # logical |0_L> preparation
    prepare_logical_zero()

    qml.Barrier(wires=DATA_WIRES)

    # optional initial data error
    for p, w in zip(initial_error_list, initial_error_wires):
        apply_pauli(p, w)

    # repeated syndrome extraction
    for t in range(n_rounds):
        round_record = syndrome_round_with_fault_schedule(
            current_round=t,
            fault_schedule=fault_schedule,
        )
        measurement_record.extend(round_record)
        qml.Barrier()

    return [qml.sample(m) for m in measurement_record]

qml.draw_mpl(repeated_syndrome_with_fault_schedule, show_all_wires=True)(
    fault_schedule=[
        {"round": 1, "wires": [0, 9], "error": ["X", "Z"]},
        {"round": 3, "wires": [4, 12], "error": ["X", "X"]},
    ],
    initial_error_list=[],
    initial_error_wires=[],
)

/Users/jhan/Library/Mobile Documents/com~apple~CloudDocs/ETRI/연구/Correlated Noise Estimation by Syndrome Measurement/syndrome_env/lib/python3.14/site-packages/pennylane/devices/device_api.py:201: PennyLaneDeprecationWarning: Setting shots on device is deprecated. Please use the `set_shots` transform on the respective QNode instead.
  warnings.warn(


(<Figure size 21100x2650 with 1 Axes>, <Axes: >)

In [22]:
# --------------------------------------------------
# Example usage
# --------------------------------------------------

fault_schedule = [
    {
        "round": 1,
        "wires": [0, 9],
        "error": ["X", "Z"],
    },
    # {
    #     "round": 3,
    #     "wires": [4, 12],
    #     "error": ["X", "X"],
    # },
]

raw = repeated_syndrome_with_fault_schedule(
    fault_schedule=fault_schedule,
)

raw = np.array(raw)

print("raw shape:", raw.shape)
# expected: (n_rounds * 8, shots)

syndrome = raw.reshape(n_rounds, 8, shots)
syndrome = np.moveaxis(syndrome, 2, 0)

print("syndrome shape:", syndrome.shape)
# expected: (shots, n_rounds, 8)

detection_events = syndrome[:, 1:, :] ^ syndrome[:, :-1, :]

print("detection_events shape:", detection_events.shape)
# expected: (shots, n_rounds - 1, 8)

print("first shot syndrome:")
print(syndrome[0])

print("first shot detection events:")
print(detection_events[0])

raw shape: (32,)
syndrome shape: (1, 4, 8)
detection_events shape: (1, 3, 8)
first shot syndrome:
[[0 0 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 0]
 [1 0 0 0 1 0 0 0]
 [1 0 0 0 1 0 0 0]]
first shot detection events:
[[0 0 0 0 1 0 0 0]
 [1 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]]
